# Evaluate One Real Validation Sample

Point this notebook at one real image and one matching label JSON, then run the local OCR/LayoutLMv3 pipelines against that sample.

This notebook is rotation-aware. By default it uses the saved `rotation` from the label JSON. You can also force a manual override in the config cell.

In [18]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint

import torch
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT.parent / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")

FINE_TUNE_DIR = REPO_ROOT / "src" / "fine_tune_LayoutLMv3"
if str(FINE_TUNE_DIR) not in sys.path:
    sys.path.insert(0, str(FINE_TUNE_DIR))

from evaluate_local_validation import compute_sequence_metrics
from experiment_utils import (
    assign_labels_to_ocr_tokens,
    compute_ocr_metrics,
    load_model,
    load_processor,
    normalize_bbox,
    predict_word_labels,
    run_ocr,
)

IMAGE_PATH = REPO_ROOT / "real_validation" / "images" / "3.png"
LABEL_PATH = REPO_ROOT / "real_validation" / "labels" / "3.json"
BASELINE_MODEL_DIR = REPO_ROOT / "src" / "fine_tune_LayoutLMv3" / "models" / "layoutlmv3-funsd"
FINETUNED_MODEL_DIR = REPO_ROOT / "runs" / "layoutlmv3-funsd-tesseract-seed42" / "best_model"
OUTPUT_DIR = REPO_ROOT / "runs" / "single_sample_eval"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Use None to respect the saved rotation in the label JSON.
# Set to 0, 90, 180, or 270 to override manually.
ROTATION_OVERRIDE = None

print({
    "image": str(IMAGE_PATH),
    "label": str(LABEL_PATH),
    "baseline_model": str(BASELINE_MODEL_DIR),
    "finetuned_model": str(FINETUNED_MODEL_DIR),
    "device": DEVICE,
    "rotation_override": ROTATION_OVERRIDE,
})

{'image': '/home/chaot/auto_doc_ai/real_validation/images/3.png', 'label': '/home/chaot/auto_doc_ai/real_validation/labels/3.json', 'baseline_model': '/home/chaot/auto_doc_ai/src/fine_tune_LayoutLMv3/models/layoutlmv3-funsd', 'finetuned_model': '/home/chaot/auto_doc_ai/runs/layoutlmv3-funsd-tesseract-seed42/best_model', 'device': 'cuda', 'rotation_override': None}


In [19]:
def get_saved_rotation(label_path: Path) -> int:
    with open(label_path) as fh:
        payload = json.load(fh)
    return int(payload.get("rotation", 0)) % 360


def rotate_normalized_bbox(bbox: list[int], rotation: int) -> list[int]:
    rotation = rotation % 360
    if rotation == 0:
        return [int(v) for v in bbox]

    x1, y1, x2, y2 = bbox
    corners = [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]

    def rotate_point(x: int, y: int) -> tuple[int, int]:
        if rotation == 90:
            return 1000 - y, x
        if rotation == 180:
            return 1000 - x, 1000 - y
        if rotation == 270:
            return y, 1000 - x
        raise ValueError(f"Unsupported rotation: {rotation}")

    rotated = [rotate_point(x, y) for x, y in corners]
    xs = [point[0] for point in rotated]
    ys = [point[1] for point in rotated]
    return [min(xs), min(ys), max(xs), max(ys)]


def prepare_image_for_rotation(image_path: Path, rotation: int, output_dir: Path) -> Path:
    rotation = rotation % 360
    if rotation == 0:
        return image_path

    prepared_dir = output_dir / "prepared_inputs"
    prepared_dir.mkdir(parents=True, exist_ok=True)
    rotated_path = prepared_dir / f"{image_path.stem}_rot{rotation}.png"
    with Image.open(image_path) as image:
        rotated = image.rotate(-rotation, expand=True)
        rotated.save(rotated_path)
    return rotated_path


def load_single_truth(label_path: Path, image_path: Path, rotation: int = 0) -> dict:
    with open(label_path) as fh:
        payload = json.load(fh)

    image_size = Image.open(image_path).size

    if "annotations" in payload and "words" in payload["annotations"]:
        from experiment_utils import build_funsd_word_labels

        words = payload["annotations"]["words"]
        bboxes = [
            [int(v) for v in word["normalized_bbox"]]
            if "normalized_bbox" in word
            else normalize_bbox(word["bbox"], *image_size)
            for word in words
        ]
        if rotation % 360 != 0:
            bboxes = [rotate_normalized_bbox(bbox, rotation) for bbox in bboxes]
        return {
            "words": [word["text"] for word in words],
            "bboxes": bboxes,
            "labels": build_funsd_word_labels(payload),
        }

    token_entries = payload.get("tokens") or payload.get("predictions") or []
    if payload.get("image_size"):
        image_size = tuple(payload["image_size"])

    words = []
    bboxes = []
    labels = []
    for token in token_entries:
        bbox = [float(v) for v in token.get("bbox", token.get("normalized_bbox", []))]
        if len(bbox) != 4:
            continue

        if all(0 <= v <= 1000 for v in bbox):
            normalized_bbox = [int(v) for v in bbox]
        else:
            normalized_bbox = normalize_bbox(bbox, *image_size)
        if rotation % 360 != 0:
            normalized_bbox = rotate_normalized_bbox(normalized_bbox, rotation)

        raw_label = str(token.get("label", "")).strip().upper() or "O"
        if raw_label in {"HEADER", "QUESTION", "ANSWER"}:
            raw_label = f"B-{raw_label}"
        if raw_label == "OTHER":
            raw_label = "O"

        words.append(str(token.get("text", "")))
        bboxes.append(normalized_bbox)
        labels.append(raw_label)

    return {
        "words": words,
        "bboxes": bboxes,
        "labels": labels,
    }


def get_model_bundle(model_dir: Path, cache: dict) -> tuple:
    key = str(model_dir.resolve())
    if key not in cache:
        processor = load_processor(model_dir)
        model = load_model(model_dir)
        model.to(DEVICE)
        cache[key] = (processor, model)
    return cache[key]


def evaluate_layoutlm_sample(
    image_path: Path,
    label_path: Path,
    model_dir: Path,
    ocr_source: str,
    model_cache: dict,
    rotation: int,
) -> dict:
    effective_image_path = prepare_image_for_rotation(image_path, rotation, OUTPUT_DIR)
    truth = load_single_truth(label_path, image_path, rotation=rotation)
    processor, model = get_model_bundle(model_dir, model_cache)

    if ocr_source == "ground_truth":
        words = truth["words"]
        boxes = truth["bboxes"]
        true_labels = truth["labels"]
    else:
        words, boxes, image_size = run_ocr(effective_image_path, ocr_source)
        if not words:
            return {
                "pipeline": f"{ocr_source} -> {model_dir.name}",
                "status": "skipped",
                "reason": "no_ocr_tokens",
                "rotation": rotation,
            }
        true_labels = assign_labels_to_ocr_tokens(
            boxes,
            truth["bboxes"],
            truth["labels"],
            image_size=image_size,
        )

    pred_labels = predict_word_labels(
        processor=processor,
        model=model,
        image_path=effective_image_path,
        words=words,
        boxes=boxes,
        device=DEVICE,
    )

    predictions = [
        {
            "token_index": idx,
            "text": word,
            "bbox": box,
            "true_label": true_label,
            "pred_label": pred_label,
        }
        for idx, (word, box, true_label, pred_label) in enumerate(zip(words, boxes, true_labels, pred_labels))
    ]

    result = {
        "pipeline": f"{ocr_source} -> {model_dir.name}",
        "status": "ok",
        "model_dir": str(model_dir),
        "ocr_source": ocr_source,
        "rotation": rotation,
        "effective_image_path": str(effective_image_path),
        "layoutlm_metrics": compute_sequence_metrics([true_labels], [pred_labels]),
        "predictions": predictions,
        "prediction_preview": predictions[:40],
    }
    if ocr_source != "ground_truth":
        result["ocr_metrics"] = compute_ocr_metrics(
            [{"image_path": effective_image_path, "words": truth["words"]}],
            ocr_source=ocr_source,
        )
    return result


def evaluate_ocr_only(image_path: Path, label_path: Path, ocr_source: str, rotation: int) -> dict:
    effective_image_path = prepare_image_for_rotation(image_path, rotation, OUTPUT_DIR)
    truth = load_single_truth(label_path, image_path, rotation=rotation)
    return {
        "pipeline": f"{ocr_source} only",
        "status": "ok",
        "ocr_source": ocr_source,
        "rotation": rotation,
        "effective_image_path": str(effective_image_path),
        "ocr_metrics": compute_ocr_metrics(
            [{"image_path": effective_image_path, "words": truth["words"]}],
            ocr_source=ocr_source,
        ),
    }


assert IMAGE_PATH.exists(), IMAGE_PATH
assert LABEL_PATH.exists(), LABEL_PATH
assert BASELINE_MODEL_DIR.exists(), BASELINE_MODEL_DIR
assert FINETUNED_MODEL_DIR.exists(), FINETUNED_MODEL_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EFFECTIVE_ROTATION = get_saved_rotation(LABEL_PATH) if ROTATION_OVERRIDE is None else int(ROTATION_OVERRIDE) % 360
EFFECTIVE_IMAGE_PATH = prepare_image_for_rotation(IMAGE_PATH, EFFECTIVE_ROTATION, OUTPUT_DIR)
print({
    "effective_rotation": EFFECTIVE_ROTATION,
    "effective_image_path": str(EFFECTIVE_IMAGE_PATH),
})

{'effective_rotation': 90, 'effective_image_path': '/home/chaot/auto_doc_ai/runs/single_sample_eval/prepared_inputs/3_rot90.png'}


In [20]:
model_cache = {}
results = []

results.append(evaluate_ocr_only(IMAGE_PATH, LABEL_PATH, "tesseract", EFFECTIVE_ROTATION))

pipelines = [
    ("baseline_ground_truth", BASELINE_MODEL_DIR, "ground_truth"),
    ("finetuned_ground_truth", FINETUNED_MODEL_DIR, "ground_truth"),
    ("baseline_tesseract", BASELINE_MODEL_DIR, "tesseract"),
    ("finetuned_tesseract", FINETUNED_MODEL_DIR, "tesseract"),
]

try:
    _ = run_ocr(EFFECTIVE_IMAGE_PATH, "paddleocr")
    results.append(evaluate_ocr_only(IMAGE_PATH, LABEL_PATH, "paddleocr", EFFECTIVE_ROTATION))
    pipelines.extend([
        ("baseline_paddleocr", BASELINE_MODEL_DIR, "paddleocr"),
        ("finetuned_paddleocr", FINETUNED_MODEL_DIR, "paddleocr"),
    ])
    print("PaddleOCR available; including Paddle pipelines.")
except Exception as exc:
    print(f"Skipping PaddleOCR pipelines: {exc}")

for name, model_dir, ocr_source in pipelines:
    print(f"Running {name}...")
    result = evaluate_layoutlm_sample(
        image_path=IMAGE_PATH,
        label_path=LABEL_PATH,
        model_dir=model_dir,
        ocr_source=ocr_source,
        model_cache=model_cache,
        rotation=EFFECTIVE_ROTATION,
    )
    result["name"] = name
    results.append(result)

for result in results:
    output_name = result.get("name") or result["pipeline"].replace(" ", "_").replace("->", "to")
    with open(OUTPUT_DIR / f"{output_name}.json", "w") as fh:
        json.dump(result, fh, indent=2)

summary = []
for result in results:
    row = {
        "pipeline": result["pipeline"],
        "rotation": result.get("rotation"),
        "status": result["status"],
    }
    ocr_metrics = result.get("ocr_metrics")
    if ocr_metrics:
        row.update({
            "word_error_rate": round(ocr_metrics["word_error_rate"], 4),
            "char_error_rate": round(ocr_metrics["char_error_rate"], 4),
            "bag_token_f1": round(ocr_metrics["bag_token_f1"], 4),
        })
    if result["status"] == "ok" and "layoutlm_metrics" in result:
        metrics = result["layoutlm_metrics"]
        row.update({
            "token_accuracy": round(metrics["token_accuracy"], 4),
            "entity_f1": round(metrics["entity_f1"], 4),
            "seqeval_f1": round(metrics["seqeval_f1"], 4),
        })
    elif result["status"] != "ok":
        row["reason"] = result.get("reason")
    summary.append(row)

pprint(summary)
print(f"Saved per-pipeline outputs to: {OUTPUT_DIR}")

Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.


Skipping PaddleOCR pipelines: In user code:


    ResourceExhaustedError: Fail to alloc memory of 62373888000 size, error code is 12.
      [Hint: Expected error == 0, but received error:12 != 0:0.] (at /paddle/paddle/phi/core/memory/allocation/cpu_allocator.cc:44)
      [operator < pd_kernel.phi_kernel > error]
Running baseline_ground_truth...


Loading weights: 100%|██████████| 214/214 [00:00<00:00, 1933.66it/s]


Running finetuned_ground_truth...


Loading weights: 100%|██████████| 214/214 [00:00<00:00, 1711.62it/s]


Running baseline_tesseract...
Running finetuned_tesseract...
[{'bag_token_f1': 0.6879,
  'char_error_rate': 0.7437,
  'pipeline': 'tesseract only',
  'rotation': 90,
  'status': 'ok',
  'word_error_rate': 1.0307},
 {'entity_f1': 0.2077,
  'pipeline': 'ground_truth -> layoutlmv3-funsd',
  'rotation': 90,
  'seqeval_f1': np.float64(0.2652),
  'status': 'ok',
  'token_accuracy': 0.2243},
 {'entity_f1': 0.2597,
  'pipeline': 'ground_truth -> best_model',
  'rotation': 90,
  'seqeval_f1': np.float64(0.37),
  'status': 'ok',
  'token_accuracy': 0.2757},
 {'bag_token_f1': 0.6879,
  'char_error_rate': 0.7437,
  'entity_f1': 0.2148,
  'pipeline': 'tesseract -> layoutlmv3-funsd',
  'rotation': 90,
  'seqeval_f1': np.float64(0.1986),
  'status': 'ok',
  'token_accuracy': 0.2653,
  'word_error_rate': 1.0307},
 {'bag_token_f1': 0.6879,
  'char_error_rate': 0.7437,
  'entity_f1': 0.2992,
  'pipeline': 'tesseract -> best_model',
  'rotation': 90,
  'seqeval_f1': np.float64(0.246),
  'status': 'ok',
 

In [21]:
import pandas as pd

def load_result(name: str) -> dict:
    with open(OUTPUT_DIR / f"{name}.json") as fh:
        return json.load(fh)

tesseract_only = load_result("tesseract_only")
baseline_tesseract = load_result("baseline_tesseract")
finetuned_tesseract = load_result("finetuned_tesseract")

focus_metrics_df = pd.DataFrame([
    {
        "pipeline": "tesseract OCR quality",
        "word_error_rate": tesseract_only["ocr_metrics"]["word_error_rate"],
        "char_error_rate": tesseract_only["ocr_metrics"]["char_error_rate"],
        "bag_token_f1": tesseract_only["ocr_metrics"]["bag_token_f1"],
    },
    {
        "pipeline": "baseline tesseract -> LayoutLMv3",
        "token_accuracy": baseline_tesseract["layoutlm_metrics"]["token_accuracy"],
        "entity_f1": baseline_tesseract["layoutlm_metrics"]["entity_f1"],
        "seqeval_f1": baseline_tesseract["layoutlm_metrics"]["seqeval_f1"],
        "word_error_rate": baseline_tesseract["ocr_metrics"]["word_error_rate"],
        "char_error_rate": baseline_tesseract["ocr_metrics"]["char_error_rate"],
        "bag_token_f1": baseline_tesseract["ocr_metrics"]["bag_token_f1"],
    },
    {
        "pipeline": "fine-tuned tesseract -> LayoutLMv3",
        "token_accuracy": finetuned_tesseract["layoutlm_metrics"]["token_accuracy"],
        "entity_f1": finetuned_tesseract["layoutlm_metrics"]["entity_f1"],
        "seqeval_f1": finetuned_tesseract["layoutlm_metrics"]["seqeval_f1"],
        "word_error_rate": finetuned_tesseract["ocr_metrics"]["word_error_rate"],
        "char_error_rate": finetuned_tesseract["ocr_metrics"]["char_error_rate"],
        "bag_token_f1": finetuned_tesseract["ocr_metrics"]["bag_token_f1"],
    },
]).round(4)
display(focus_metrics_df)

baseline_pred_df = pd.DataFrame(baseline_tesseract["predictions"])[["token_index", "text", "true_label", "pred_label"]]
finetuned_pred_df = pd.DataFrame(finetuned_tesseract["predictions"])[["token_index", "text", "true_label", "pred_label"]]

prediction_compare_df = baseline_pred_df.rename(
    columns={
        "text": "ocr_text",
        "true_label": "true_label",
        "pred_label": "baseline_pred",
    }
).merge(
    finetuned_pred_df.rename(
        columns={
            "text": "finetuned_text",
            "true_label": "finetuned_true_label",
            "pred_label": "finetuned_pred",
        }
    ),
    on="token_index",
    how="outer",
)
prediction_compare_df["ocr_text"] = prediction_compare_df["ocr_text"].fillna(prediction_compare_df["finetuned_text"])
prediction_compare_df["true_label"] = prediction_compare_df["true_label"].fillna(prediction_compare_df["finetuned_true_label"])
prediction_compare_df = prediction_compare_df[["token_index", "ocr_text", "true_label", "baseline_pred", "finetuned_pred"]]
display(prediction_compare_df.head(60))

,pipeline,word_error_rate,char_error_rate,bag_token_f1,token_accuracy,entity_f1,seqeval_f1
0,tesseract OCR quality,1.0307,0.7437,0.6879,NaN,NaN,NaN
1,baseline tesseract -> LayoutLMv3,1.0307,0.7437,0.6879,0.2653,0.2148,0.1986
2,fine-tuned tesseract -> LayoutLMv3,1.0307,0.7437,0.6879,0.4082,0.2992,0.2460


,token_index,ocr_text,true_label,baseline_pred,finetuned_pred
0,0,0102077,O,O,O
1,1,7,O,O,O
2,2,rl,O,B-ANSWER,O
3,3,al,O,B-ANSWER,O
4,4,(Gi,O,O,O
5,5,ai,O,O,O
6,6,s<],O,O,O
7,7,i,O,O,O
8,8,~,B-HEADER,B-QUESTION,B-QUESTION
9,9,TVTLE,B-QUESTION,B-QUESTION,B-QUESTION
